---
## The database you will quiz on

```
departments(dept_id, dept_name, hod)
students   (student_id, name, age, gender, city, dept_id, enroll_year)
subjects   (subject_id, subject_name, dept_id)
marks      (mark_id, student_id, subject_id, marks)
```

Deliberate "trap" data, so good candidates get tested on edge cases:
- **Civil** department has no students
- **Pooja** has no department (`dept_id` is NULL)
- **Nikhil** has no marks
- Two marks tie at **72** (useful for RANK vs DENSE_RANK)

## Step 1: Create the database
Run this cell once. It builds the tables, inserts the sample data and defines the `ask()` helper that prints each question, model answer and output.

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# Create an in-memory database
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# ---------- Create tables ----------
cur.executescript("""
DROP TABLE IF EXISTS marks;
DROP TABLE IF EXISTS subjects;
DROP TABLE IF EXISTS students;
DROP TABLE IF EXISTS departments;

CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL,
    hod       TEXT
);

CREATE TABLE students (
    student_id  INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    age         INTEGER,
    gender      TEXT,
    city        TEXT,
    dept_id     INTEGER,
    enroll_year INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);

CREATE TABLE subjects (
    subject_id   INTEGER PRIMARY KEY,
    subject_name TEXT NOT NULL,
    dept_id      INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);

CREATE TABLE marks (
    mark_id    INTEGER PRIMARY KEY,
    student_id INTEGER,
    subject_id INTEGER,
    marks      INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (subject_id) REFERENCES subjects(subject_id)
);
""")

# ---------- Insert sample data ----------
departments = [
    (1, "Computer Science", "Dr. Rao"),
    (2, "Electronics",      "Dr. Iyer"),
    (3, "Mechanical",       "Dr. Khan"),
    (4, "Civil",            "Dr. Menon"),   # no students
]

students = [
    (1,  "Aarav",   20, "M", "Hyderabad", 1,    2023),
    (2,  "Priya",   21, "F", "Chennai",   1,    2022),
    (3,  "Rohan",   19, "M", "Bengaluru", 2,    2024),
    (4,  "Sneha",   22, "F", "Hyderabad", 2,    2021),
    (5,  "Karthik", 20, "M", "Chennai",   3,    2023),
    (6,  "Ananya",  21, "F", "Bengaluru", 1,    2022),
    (7,  "Vikram",  23, "M", "Mumbai",    3,    2021),
    (8,  "Meera",   19, "F", "Hyderabad", 2,    2024),
    (9,  "Arjun",   22, "M", "Pune",      1,    2021),
    (10, "Divya",   20, "F", "Chennai",   2,    2023),
    (11, "Suresh",  24, "M", "Mumbai",    3,    2020),
    (12, "Pooja",   21, "F", "Pune",      None, 2022),   # no department
    (13, "Nikhil",  20, "M", "Hyderabad", 1,    2023),   # no marks
]

subjects = [
    (101, "Database Systems",  1),
    (102, "Data Structures",   1),
    (103, "Digital Circuits",  2),
    (104, "Signals",           2),
    (105, "Thermodynamics",    3),
]

marks = [
    (1, 1, 101, 85), (2, 1, 102, 78),
    (3, 2, 101, 92), (4, 2, 102, 88),
    (5, 3, 103, 67), (6, 3, 104, 72),
    (7, 4, 103, 90), (8, 4, 104, 81),
    (9, 5, 105, 55),
    (10, 6, 101, 76), (11, 6, 102, 95),
    (12, 7, 105, 48),
    (13, 8, 103, 83), (14, 8, 104, 79),
    (15, 9, 101, 64), (16, 9, 102, 72),
    (17, 10, 103, 74), (18, 10, 104, 86),
    (19, 11, 105, 39),
    (20, 12, 101, 58),
]

cur.executemany("INSERT INTO departments VALUES (?, ?, ?)", departments)
cur.executemany("INSERT INTO students    VALUES (?, ?, ?, ?, ?, ?, ?)", students)
cur.executemany("INSERT INTO subjects    VALUES (?, ?, ?)", subjects)
cur.executemany("INSERT INTO marks       VALUES (?, ?, ?, ?)", marks)
conn.commit()

# ---------- Helper functions used in the whole notebook ----------
question_number = 0

def run_query(sql):
    """Run a SQL query and return the result as a Pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

def ask(question, sql, level="Easy", follow_up="", look_for=""):
    """Show one interview question: the question, model answer, output, follow-up and what to look for."""
    global question_number
    question_number += 1
    icon = {"Easy": "🟢", "Medium": "🟡", "Hard": "🔴"}.get(level, "⚪")
    print("=" * 74)
    print(f"❓ Q{question_number}  {icon} {level}")
    print(question)
    print()
    print("✅ MODEL ANSWER")
    print(sql.strip())
    print()
    print("📊 EXPECTED OUTPUT")
    display(run_query(sql))
    if follow_up:
        print("🔁 FOLLOW-UP:", follow_up)
    if look_for:
        print("👀 LOOK FOR:", look_for)
    print()

print("✅ Database created and sample data inserted!")

✅ Database created and sample data inserted!


### Preview the tables (useful to share on screen with the candidate)

In [ ]:
for table in ["departments", "students", "subjects", "marks"]:
    print(f"\n📋 Table: {table}")
    display(run_query(f"SELECT * FROM {table}"))


📋 Table: departments


,dept_id,dept_name,hod
0,1,Computer Science,Dr. Rao
1,2,Electronics,Dr. Iyer
2,3,Mechanical,Dr. Khan
3,4,Civil,Dr. Menon



📋 Table: students


,student_id,name,age,gender,city,dept_id,enroll_year
0,1,Aarav,20,M,Hyderabad,1.0,2023
1,2,Priya,21,F,Chennai,1.0,2022
2,3,Rohan,19,M,Bengaluru,2.0,2024
3,4,Sneha,22,F,Hyderabad,2.0,2021
4,5,Karthik,20,M,Chennai,3.0,2023
5,6,Ananya,21,F,Bengaluru,1.0,2022
6,7,Vikram,23,M,Mumbai,3.0,2021
7,8,Meera,19,F,Hyderabad,2.0,2024
8,9,Arjun,22,M,Pune,1.0,2021
9,10,Divya,20,F,Chennai,2.0,2023



📋 Table: subjects


,subject_id,subject_name,dept_id
0,101,Database Systems,1
1,102,Data Structures,1
2,103,Digital Circuits,2
3,104,Signals,2
4,105,Thermodynamics,3



📋 Table: marks


,mark_id,student_id,subject_id,marks
0,1,1,101,85
1,2,1,102,78
2,3,2,101,92
3,4,2,102,88
4,5,3,103,67
5,6,3,104,72
6,7,4,103,90
7,8,4,104,81
8,9,5,105,55
9,10,6,101,76


---
# 1️⃣ SELECT
**What you are testing:** column selection, aliases, expressions, `CASE`.

### 🗣️ Concept question
**Q: What does `SELECT *` do, and why do many teams avoid it in production code?**
> **Expected answer:** It returns every column. It is fine for exploring, but in production it wastes bandwidth, can break code when columns are added or reordered, and stops the database from using covering indexes.

In [ ]:
ask("Show each student's name and city, with the column headers 'Student' and 'Hometown'.",
"""
SELECT name AS Student, city AS Hometown
FROM students;
""",
level="Easy",
follow_up="Ask: What if the alias needs a space, like 'Home Town'?  Expect: wrap it in double quotes, e.g. AS \"Home Town\".",
look_for="Uses AS (or a bare alias). Does not use SELECT *.")

ask("Create a column that shows how old each student will be in 5 years.",
"""
SELECT name, age, age + 5 AS age_in_5_years
FROM students;
""",
level="Easy",
look_for="Knows SELECT can contain expressions, not just columns.")

ask("Label every mark as 'Pass' if it is 40 or more, otherwise 'Fail'.",
"""
SELECT mark_id, marks,
       CASE WHEN marks >= 40 THEN 'Pass' ELSE 'Fail' END AS result
FROM marks;
""",
level="Medium",
follow_up="Ask: Add a 'Distinction' label for 85 and above.  Expect: a first WHEN marks >= 85 branch BEFORE the pass branch, because CASE stops at the first match.",
look_for="Uses CASE WHEN ... THEN ... ELSE ... END. Watch for branch ORDER mistakes.")

❓ Q1  🟢 Easy
Show each student's name and city, with the column headers 'Student' and 'Hometown'.

✅ MODEL ANSWER
SELECT name AS Student, city AS Hometown
FROM students;

📊 EXPECTED OUTPUT


,Student,Hometown
0,Aarav,Hyderabad
1,Priya,Chennai
2,Rohan,Bengaluru
3,Sneha,Hyderabad
4,Karthik,Chennai
5,Ananya,Bengaluru
6,Vikram,Mumbai
7,Meera,Hyderabad
8,Arjun,Pune
9,Divya,Chennai


🔁 FOLLOW-UP: Ask: What if the alias needs a space, like 'Home Town'?  Expect: wrap it in double quotes, e.g. AS "Home Town".
👀 LOOK FOR: Uses AS (or a bare alias). Does not use SELECT *.

❓ Q2  🟢 Easy
Create a column that shows how old each student will be in 5 years.

✅ MODEL ANSWER
SELECT name, age, age + 5 AS age_in_5_years
FROM students;

📊 EXPECTED OUTPUT


,name,age,age_in_5_years
0,Aarav,20,25
1,Priya,21,26
2,Rohan,19,24
3,Sneha,22,27
4,Karthik,20,25
5,Ananya,21,26
6,Vikram,23,28
7,Meera,19,24
8,Arjun,22,27
9,Divya,20,25


👀 LOOK FOR: Knows SELECT can contain expressions, not just columns.

❓ Q3  🟡 Medium
Label every mark as 'Pass' if it is 40 or more, otherwise 'Fail'.

✅ MODEL ANSWER
SELECT mark_id, marks,
       CASE WHEN marks >= 40 THEN 'Pass' ELSE 'Fail' END AS result
FROM marks;

📊 EXPECTED OUTPUT


,mark_id,marks,result
0,1,85,Pass
1,2,78,Pass
2,3,92,Pass
3,4,88,Pass
4,5,67,Pass
5,6,72,Pass
6,7,90,Pass
7,8,81,Pass
8,9,55,Pass
9,10,76,Pass


🔁 FOLLOW-UP: Ask: Add a 'Distinction' label for 85 and above.  Expect: a first WHEN marks >= 85 branch BEFORE the pass branch, because CASE stops at the first match.
👀 LOOK FOR: Uses CASE WHEN ... THEN ... ELSE ... END. Watch for branch ORDER mistakes.



---
# 2️⃣ WHERE
**What you are testing:** filtering, comparison operators, and the classic **NULL trap**.

### 🗣️ Concept question
**Q: Why does `WHERE dept_id = NULL` return no rows? How do you fix it?**
> **Expected answer:** `NULL` means "unknown", and any comparison with NULL (even `NULL = NULL`) is unknown, not true, so no row passes. Use `IS NULL` / `IS NOT NULL`.

In [ ]:
ask("List students from Chennai who are older than 20.",
"""
SELECT name, age, city
FROM students
WHERE city = 'Chennai' AND age > 20;
""",
level="Easy",
look_for="Correct string quoting. Uses > not >=.")

ask("Find students who have not been assigned to any department.",
"""
SELECT name, dept_id
FROM students
WHERE dept_id IS NULL;
""",
level="Easy",
follow_up="Ask: What would happen with WHERE dept_id = NULL?  Expect: zero rows.",
look_for="IS NULL (not = NULL).")

ask("Show every student who is NOT in department 1, including students with no department.",
"""
SELECT name, dept_id
FROM students
WHERE dept_id <> 1 OR dept_id IS NULL;
""",
level="Medium",
follow_up="Ask: What does WHERE dept_id <> 1 alone return?  Expect: Pooja is missing, because NULL <> 1 is unknown, not true.",
look_for="Remembers that NULL rows silently disappear from <> filters. This question separates careful candidates from careless ones.")

❓ Q4  🟢 Easy
List students from Chennai who are older than 20.

✅ MODEL ANSWER
SELECT name, age, city
FROM students
WHERE city = 'Chennai' AND age > 20;

📊 EXPECTED OUTPUT


,name,age,city
0,Priya,21,Chennai


👀 LOOK FOR: Correct string quoting. Uses > not >=.

❓ Q5  🟢 Easy
Find students who have not been assigned to any department.

✅ MODEL ANSWER
SELECT name, dept_id
FROM students
WHERE dept_id IS NULL;

📊 EXPECTED OUTPUT


,name,dept_id
0,Pooja,None


🔁 FOLLOW-UP: Ask: What would happen with WHERE dept_id = NULL?  Expect: zero rows.
👀 LOOK FOR: IS NULL (not = NULL).

❓ Q6  🟡 Medium
Show every student who is NOT in department 1, including students with no department.

✅ MODEL ANSWER
SELECT name, dept_id
FROM students
WHERE dept_id <> 1 OR dept_id IS NULL;

📊 EXPECTED OUTPUT


,name,dept_id
0,Rohan,2.0
1,Sneha,2.0
2,Karthik,3.0
3,Vikram,3.0
4,Meera,2.0
5,Divya,2.0
6,Suresh,3.0
7,Pooja,NaN


🔁 FOLLOW-UP: Ask: What does WHERE dept_id <> 1 alone return?  Expect: Pooja is missing, because NULL <> 1 is unknown, not true.
👀 LOOK FOR: Remembers that NULL rows silently disappear from <> filters. This question separates careful candidates from careless ones.



---
# 3️⃣ DISTINCT
**What you are testing:** de-duplication and counting unique values.

### 🗣️ Concept question
**Q: What is the difference between `SELECT DISTINCT city` and `SELECT city ... GROUP BY city`?**
> **Expected answer:** For plain de-duplication they return the same rows. `GROUP BY` is needed when you also want aggregates (`COUNT`, `AVG`...). `DISTINCT` applies to the whole row, not just the first column.

In [ ]:
ask("Which cities do our students come from? Each city only once.",
"""
SELECT DISTINCT city
FROM students
ORDER BY city;
""",
level="Easy",
look_for="Uses DISTINCT. Bonus for adding ORDER BY.")

ask("How many different cities do our students come from?",
"""
SELECT COUNT(DISTINCT city) AS number_of_cities
FROM students;
""",
level="Easy",
follow_up="Ask: What does SELECT DISTINCT city, gender return?  Expect: unique (city, gender) PAIRS, not unique cities.",
look_for="COUNT(DISTINCT col), not COUNT(*) or a subquery.")

ask("How many students have at least one mark recorded?",
"""
SELECT COUNT(DISTINCT student_id) AS students_with_marks
FROM marks;
""",
level="Medium",
look_for="Realises COUNT(*) on marks would count rows (20), not students (12).")

❓ Q7  🟢 Easy
Which cities do our students come from? Each city only once.

✅ MODEL ANSWER
SELECT DISTINCT city
FROM students
ORDER BY city;

📊 EXPECTED OUTPUT


,city
0,Bengaluru
1,Chennai
2,Hyderabad
3,Mumbai
4,Pune


👀 LOOK FOR: Uses DISTINCT. Bonus for adding ORDER BY.

❓ Q8  🟢 Easy
How many different cities do our students come from?

✅ MODEL ANSWER
SELECT COUNT(DISTINCT city) AS number_of_cities
FROM students;

📊 EXPECTED OUTPUT


,number_of_cities
0,5


🔁 FOLLOW-UP: Ask: What does SELECT DISTINCT city, gender return?  Expect: unique (city, gender) PAIRS, not unique cities.
👀 LOOK FOR: COUNT(DISTINCT col), not COUNT(*) or a subquery.

❓ Q9  🟡 Medium
How many students have at least one mark recorded?

✅ MODEL ANSWER
SELECT COUNT(DISTINCT student_id) AS students_with_marks
FROM marks;

📊 EXPECTED OUTPUT


,students_with_marks
0,12


👀 LOOK FOR: Realises COUNT(*) on marks would count rows (20), not students (12).



---
# 4️⃣ ORDER BY
**What you are testing:** sorting, tie-breakers, custom ordering.

### 🗣️ Concept question
**Q: Is `ORDER BY` ascending or descending by default? Where do NULLs appear?**
> **Expected answer:** Ascending. In SQLite, NULLs are treated as the smallest value, so they come first in `ASC` and last in `DESC`. Other databases differ (PostgreSQL puts NULLs last in ASC), so use `NULLS FIRST/LAST` where supported.

In [ ]:
ask("List students from oldest to youngest. If two students are the same age, sort them by name.",
"""
SELECT name, age
FROM students
ORDER BY age DESC, name ASC;
""",
level="Easy",
look_for="Multi-column ORDER BY with a tie-breaker.")

ask("List students sorted by department id, but put students with NO department at the bottom.",
"""
SELECT name, dept_id
FROM students
ORDER BY dept_id IS NULL, dept_id, name;
""",
level="Medium",
follow_up="Ask: Why does this work?  Expect: 'dept_id IS NULL' is 0 or 1, so non-NULL rows (0) sort before NULL rows (1).",
look_for="Creative use of an expression in ORDER BY (or CASE WHEN ... IS NULL THEN 1 ELSE 0 END).")

ask("List students ordered by the length of their name, shortest first.",
"""
SELECT name, LENGTH(name) AS name_length
FROM students
ORDER BY LENGTH(name), name;
""",
level="Medium",
look_for="Knows you can sort by a function result, not only by a stored column.")

❓ Q10  🟢 Easy
List students from oldest to youngest. If two students are the same age, sort them by name.

✅ MODEL ANSWER
SELECT name, age
FROM students
ORDER BY age DESC, name ASC;

📊 EXPECTED OUTPUT


,name,age
0,Suresh,24
1,Vikram,23
2,Arjun,22
3,Sneha,22
4,Ananya,21
5,Pooja,21
6,Priya,21
7,Aarav,20
8,Divya,20
9,Karthik,20


👀 LOOK FOR: Multi-column ORDER BY with a tie-breaker.

❓ Q11  🟡 Medium
List students sorted by department id, but put students with NO department at the bottom.

✅ MODEL ANSWER
SELECT name, dept_id
FROM students
ORDER BY dept_id IS NULL, dept_id, name;

📊 EXPECTED OUTPUT


,name,dept_id
0,Aarav,1.0
1,Ananya,1.0
2,Arjun,1.0
3,Nikhil,1.0
4,Priya,1.0
5,Divya,2.0
6,Meera,2.0
7,Rohan,2.0
8,Sneha,2.0
9,Karthik,3.0


🔁 FOLLOW-UP: Ask: Why does this work?  Expect: 'dept_id IS NULL' is 0 or 1, so non-NULL rows (0) sort before NULL rows (1).
👀 LOOK FOR: Creative use of an expression in ORDER BY (or CASE WHEN ... IS NULL THEN 1 ELSE 0 END).

❓ Q12  🟡 Medium
List students ordered by the length of their name, shortest first.

✅ MODEL ANSWER
SELECT name, LENGTH(name) AS name_length
FROM students
ORDER BY LENGTH(name), name;

📊 EXPECTED OUTPUT


,name,name_length
0,Aarav,5
1,Arjun,5
2,Divya,5
3,Meera,5
4,Pooja,5
5,Priya,5
6,Rohan,5
7,Sneha,5
8,Ananya,6
9,Nikhil,6


👀 LOOK FOR: Knows you can sort by a function result, not only by a stored column.



---
# 5️⃣ LIMIT / OFFSET
**What you are testing:** Top-N queries, pagination, tie handling.

### 🗣️ Concept question
**Q: Is `LIMIT` safe to use without `ORDER BY`?**
> **Expected answer:** No. Without `ORDER BY` the row order is not guaranteed, so you get an arbitrary set of rows.

In [ ]:
ask("Who are the 3 oldest students?",
"""
SELECT name, age
FROM students
ORDER BY age DESC
LIMIT 3;
""",
level="Easy",
follow_up="Ask: Sneha and Arjun are both 22. Is this answer fair?  Expect: no, LIMIT cuts arbitrarily among ties. A better answer uses RANK/DENSE_RANK or a subquery on the 3rd highest age.",
look_for="Spots the tie problem on their own.")

ask("What is the second highest mark in the marks table?",
"""
SELECT DISTINCT marks AS second_highest_mark
FROM marks
ORDER BY marks DESC
LIMIT 1 OFFSET 1;
""",
level="Medium",
follow_up="Ask: Why DISTINCT?  Expect: if two students share the top mark, the 'second row' would just repeat the top mark.",
look_for="DISTINCT + OFFSET 1. A subquery with MAX also earns full marks.")

ask("Our app shows 4 students per page. Give me the SQL for page 3.",
"""
SELECT student_id, name
FROM students
ORDER BY student_id
LIMIT 4 OFFSET 8;
""",
level="Medium",
follow_up="Ask: What is the general formula?  Expect: OFFSET = (page - 1) * page_size.",
look_for="Correct offset arithmetic and a deterministic ORDER BY.")

❓ Q13  🟢 Easy
Who are the 3 oldest students?

✅ MODEL ANSWER
SELECT name, age
FROM students
ORDER BY age DESC
LIMIT 3;

📊 EXPECTED OUTPUT


,name,age
0,Suresh,24
1,Vikram,23
2,Sneha,22


🔁 FOLLOW-UP: Ask: Sneha and Arjun are both 22. Is this answer fair?  Expect: no, LIMIT cuts arbitrarily among ties. A better answer uses RANK/DENSE_RANK or a subquery on the 3rd highest age.
👀 LOOK FOR: Spots the tie problem on their own.

❓ Q14  🟡 Medium
What is the second highest mark in the marks table?

✅ MODEL ANSWER
SELECT DISTINCT marks AS second_highest_mark
FROM marks
ORDER BY marks DESC
LIMIT 1 OFFSET 1;

📊 EXPECTED OUTPUT


,second_highest_mark
0,92


🔁 FOLLOW-UP: Ask: Why DISTINCT?  Expect: if two students share the top mark, the 'second row' would just repeat the top mark.
👀 LOOK FOR: DISTINCT + OFFSET 1. A subquery with MAX also earns full marks.

❓ Q15  🟡 Medium
Our app shows 4 students per page. Give me the SQL for page 3.

✅ MODEL ANSWER
SELECT student_id, name
FROM students
ORDER BY student_id
LIMIT 4 OFFSET 8;

📊 EXPECTED OUTPUT


,student_id,name
0,9,Arjun
1,10,Divya
2,11,Suresh
3,12,Pooja


🔁 FOLLOW-UP: Ask: What is the general formula?  Expect: OFFSET = (page - 1) * page_size.
👀 LOOK FOR: Correct offset arithmetic and a deterministic ORDER BY.



---
# 6️⃣ AND / OR / NOT
**What you are testing:** boolean logic and operator precedence.

### 🗣️ Concept question
**Q: In `A OR B AND C`, which is evaluated first?**
> **Expected answer:** `AND` binds tighter than `OR`, so it is `A OR (B AND C)`. Always add parentheses to show your intent.

In [ ]:
# Interviewer shows this BUGGY query and asks: "This should return only FEMALE students
# from Hyderabad or Chennai. What is wrong?"
ask("🐞 Spot the bug: this query should return only female students from Hyderabad or Chennai. What is wrong with it?",
"""
SELECT name, gender, city
FROM students
WHERE gender = 'F' AND city = 'Hyderabad' OR city = 'Chennai';
""",
level="Medium",
follow_up="Notice Karthik (male) appears. Ask the candidate to fix it. Expect parentheses around the two cities (see next question).",
look_for="Explains AND-before-OR precedence without being prompted.")

ask("Fix it: female students from Hyderabad or Chennai.",
"""
SELECT name, gender, city
FROM students
WHERE gender = 'F' AND (city = 'Hyderabad' OR city = 'Chennai');
""",
level="Medium",
follow_up="Ask: Is there a shorter way to write the city check?  Expect: city IN ('Hyderabad', 'Chennai').",
look_for="Parentheses. Bonus for suggesting IN.")

ask("List students aged 20 or more who are from neither Pune nor Mumbai.",
"""
SELECT name, age, city
FROM students
WHERE age >= 20
  AND NOT (city = 'Pune' OR city = 'Mumbai');
""",
level="Medium",
follow_up="Ask: Rewrite using De Morgan's law.  Expect: city <> 'Pune' AND city <> 'Mumbai' (NOT flips OR to AND).",
look_for="Correct handling of NOT with parentheses.")

❓ Q16  🟡 Medium
🐞 Spot the bug: this query should return only female students from Hyderabad or Chennai. What is wrong with it?

✅ MODEL ANSWER
SELECT name, gender, city
FROM students
WHERE gender = 'F' AND city = 'Hyderabad' OR city = 'Chennai';

📊 EXPECTED OUTPUT


,name,gender,city
0,Priya,F,Chennai
1,Sneha,F,Hyderabad
2,Karthik,M,Chennai
3,Meera,F,Hyderabad
4,Divya,F,Chennai


🔁 FOLLOW-UP: Notice Karthik (male) appears. Ask the candidate to fix it. Expect parentheses around the two cities (see next question).
👀 LOOK FOR: Explains AND-before-OR precedence without being prompted.

❓ Q17  🟡 Medium
Fix it: female students from Hyderabad or Chennai.

✅ MODEL ANSWER
SELECT name, gender, city
FROM students
WHERE gender = 'F' AND (city = 'Hyderabad' OR city = 'Chennai');

📊 EXPECTED OUTPUT


,name,gender,city
0,Priya,F,Chennai
1,Sneha,F,Hyderabad
2,Meera,F,Hyderabad
3,Divya,F,Chennai


🔁 FOLLOW-UP: Ask: Is there a shorter way to write the city check?  Expect: city IN ('Hyderabad', 'Chennai').
👀 LOOK FOR: Parentheses. Bonus for suggesting IN.

❓ Q18  🟡 Medium
List students aged 20 or more who are from neither Pune nor Mumbai.

✅ MODEL ANSWER
SELECT name, age, city
FROM students
WHERE age >= 20
  AND NOT (city = 'Pune' OR city = 'Mumbai');

📊 EXPECTED OUTPUT


,name,age,city
0,Aarav,20,Hyderabad
1,Priya,21,Chennai
2,Sneha,22,Hyderabad
3,Karthik,20,Chennai
4,Ananya,21,Bengaluru
5,Divya,20,Chennai
6,Nikhil,20,Hyderabad


🔁 FOLLOW-UP: Ask: Rewrite using De Morgan's law.  Expect: city <> 'Pune' AND city <> 'Mumbai' (NOT flips OR to AND).
👀 LOOK FOR: Correct handling of NOT with parentheses.



---
# 7️⃣ BETWEEN
**What you are testing:** range filters and CASE-based bucketing.

### 🗣️ Concept question
**Q: Is `BETWEEN 10 AND 20` inclusive or exclusive?**
> **Expected answer:** Inclusive on both ends. It equals `>= 10 AND <= 20`. Be careful with timestamps: `BETWEEN '2024-01-01' AND '2024-01-31'` misses rows later on Jan 31.

In [ ]:
ask("Show marks that are between 60 and 80.",
"""
SELECT mark_id, student_id, marks
FROM marks
WHERE marks BETWEEN 60 AND 80
ORDER BY marks;
""",
level="Easy",
follow_up="Ask: Is a mark of exactly 60 included?  Expect: yes, BETWEEN is inclusive.",
look_for="Knows both bounds are included.")

ask("Find students aged 20 to 22 who enrolled between 2021 and 2023.",
"""
SELECT name, age, enroll_year
FROM students
WHERE age BETWEEN 20 AND 22
  AND enroll_year BETWEEN 2021 AND 2023;
""",
level="Medium",
look_for="Combines two BETWEEN conditions with AND.")

ask("Grade each mark (A: 85+, B: 70-84, C: 50-69, D: below 50) and count how many marks fall in each grade.",
"""
SELECT CASE
           WHEN marks >= 85            THEN 'A'
           WHEN marks BETWEEN 70 AND 84 THEN 'B'
           WHEN marks BETWEEN 50 AND 69 THEN 'C'
           ELSE 'D'
       END AS grade,
       COUNT(*) AS number_of_marks
FROM marks
GROUP BY grade
ORDER BY grade;
""",
level="Hard",
follow_up="Ask: Do you need the BETWEENs, given CASE stops at the first match?  Expect: no, ordered thresholds (>= 85, >= 70, >= 50) are simpler.",
look_for="Combines CASE + GROUP BY. Watch for gaps/overlaps in ranges (e.g. marks of 84.5).")

❓ Q19  🟢 Easy
Show marks that are between 60 and 80.

✅ MODEL ANSWER
SELECT mark_id, student_id, marks
FROM marks
WHERE marks BETWEEN 60 AND 80
ORDER BY marks;

📊 EXPECTED OUTPUT


,mark_id,student_id,marks
0,15,9,64
1,5,3,67
2,6,3,72
3,16,9,72
4,17,10,74
5,10,6,76
6,2,1,78
7,14,8,79


🔁 FOLLOW-UP: Ask: Is a mark of exactly 60 included?  Expect: yes, BETWEEN is inclusive.
👀 LOOK FOR: Knows both bounds are included.

❓ Q20  🟡 Medium
Find students aged 20 to 22 who enrolled between 2021 and 2023.

✅ MODEL ANSWER
SELECT name, age, enroll_year
FROM students
WHERE age BETWEEN 20 AND 22
  AND enroll_year BETWEEN 2021 AND 2023;

📊 EXPECTED OUTPUT


,name,age,enroll_year
0,Aarav,20,2023
1,Priya,21,2022
2,Sneha,22,2021
3,Karthik,20,2023
4,Ananya,21,2022
5,Arjun,22,2021
6,Divya,20,2023
7,Pooja,21,2022
8,Nikhil,20,2023


👀 LOOK FOR: Combines two BETWEEN conditions with AND.

❓ Q21  🔴 Hard
Grade each mark (A: 85+, B: 70-84, C: 50-69, D: below 50) and count how many marks fall in each grade.

✅ MODEL ANSWER
SELECT CASE
           WHEN marks >= 85            THEN 'A'
           WHEN marks BETWEEN 70 AND 84 THEN 'B'
           WHEN marks BETWEEN 50 AND 69 THEN 'C'
           ELSE 'D'
       END AS grade,
       COUNT(*) AS number_of_marks
FROM marks
GROUP BY grade
ORDER BY grade;

📊 EXPECTED OUTPUT


,grade,number_of_marks
0,A,6
1,B,8
2,C,4
3,D,2


🔁 FOLLOW-UP: Ask: Do you need the BETWEENs, given CASE stops at the first match?  Expect: no, ordered thresholds (>= 85, >= 70, >= 50) are simpler.
👀 LOOK FOR: Combines CASE + GROUP BY. Watch for gaps/overlaps in ranges (e.g. marks of 84.5).



---
# 8️⃣ IN
**What you are testing:** list membership, `IN` with subqueries, and the **NOT IN + NULL trap**.

### 🗣️ Concept question
**Q: What is the danger of `NOT IN` when the list contains a NULL?**
> **Expected answer:** `x NOT IN (1, 2, NULL)` becomes `x<>1 AND x<>2 AND x<>NULL`, and the last part is unknown, so the whole condition is never true. The query returns **no rows**. Use `NOT EXISTS` or filter NULLs out of the subquery.

In [ ]:
ask("List students who live in Pune or Mumbai.",
"""
SELECT name, city
FROM students
WHERE city IN ('Pune', 'Mumbai');
""",
level="Easy",
look_for="Uses IN instead of a chain of ORs.")

ask("List students whose department is headed by Dr. Rao or Dr. Iyer.",
"""
SELECT name, dept_id
FROM students
WHERE dept_id IN (SELECT dept_id
                  FROM departments
                  WHERE hod IN ('Dr. Rao', 'Dr. Iyer'));
""",
level="Medium",
follow_up="Ask: How else could you write this?  Expect: a JOIN between students and departments.",
look_for="IN with a subquery. Bonus for mentioning the JOIN alternative.")

ask("🐞 Demonstrate the NULL trap: find students whose dept_id is NOT IN (1, 2, NULL).",
"""
SELECT name, dept_id
FROM students
WHERE dept_id NOT IN (1, 2, NULL);
""",
level="Hard",
follow_up="The result is EMPTY even though Karthik (dept 3) should qualify. Ask the candidate why.  Expect: NULL in the list makes every comparison unknown.",
look_for="Can explain three-valued logic (TRUE / FALSE / UNKNOWN).")

❓ Q22  🟢 Easy
List students who live in Pune or Mumbai.

✅ MODEL ANSWER
SELECT name, city
FROM students
WHERE city IN ('Pune', 'Mumbai');

📊 EXPECTED OUTPUT


,name,city
0,Vikram,Mumbai
1,Arjun,Pune
2,Suresh,Mumbai
3,Pooja,Pune


👀 LOOK FOR: Uses IN instead of a chain of ORs.

❓ Q23  🟡 Medium
List students whose department is headed by Dr. Rao or Dr. Iyer.

✅ MODEL ANSWER
SELECT name, dept_id
FROM students
WHERE dept_id IN (SELECT dept_id
                  FROM departments
                  WHERE hod IN ('Dr. Rao', 'Dr. Iyer'));

📊 EXPECTED OUTPUT


,name,dept_id
0,Aarav,1
1,Priya,1
2,Rohan,2
3,Sneha,2
4,Ananya,1
5,Meera,2
6,Arjun,1
7,Divya,2
8,Nikhil,1


🔁 FOLLOW-UP: Ask: How else could you write this?  Expect: a JOIN between students and departments.
👀 LOOK FOR: IN with a subquery. Bonus for mentioning the JOIN alternative.

❓ Q24  🔴 Hard
🐞 Demonstrate the NULL trap: find students whose dept_id is NOT IN (1, 2, NULL).

✅ MODEL ANSWER
SELECT name, dept_id
FROM students
WHERE dept_id NOT IN (1, 2, NULL);

📊 EXPECTED OUTPUT


,name,dept_id


🔁 FOLLOW-UP: The result is EMPTY even though Karthik (dept 3) should qualify. Ask the candidate why.  Expect: NULL in the list makes every comparison unknown.
👀 LOOK FOR: Can explain three-valued logic (TRUE / FALSE / UNKNOWN).



---
# 9️⃣ LIKE
**What you are testing:** pattern matching with `%` and `_`.

### 🗣️ Concept question
**Q: What do `%` and `_` mean in `LIKE`? How do you search for a literal `%`?**
> **Expected answer:** `%` = any number of characters, `_` = exactly one character. To match a literal `%`, use `ESCAPE`, e.g. `LIKE '50\%' ESCAPE '\'`. Also, `LIKE` is case-insensitive for English letters in SQLite but case-sensitive in some other databases.

In [ ]:
ask("Find all students whose name starts with 'A'.",
"""
SELECT name
FROM students
WHERE name LIKE 'A%';
""",
level="Easy",
look_for="Pattern 'A%' (not '%A%').")

ask("Find students whose name has exactly 6 letters.",
"""
SELECT name
FROM students
WHERE name LIKE '______';
""",
level="Easy",
follow_up="Ask: Is there a cleaner way than six underscores?  Expect: WHERE LENGTH(name) = 6.",
look_for="Knows _ matches exactly one character. Bonus for the LENGTH alternative.")

ask("Find students whose name contains the letter 'a' at least twice.",
"""
SELECT name
FROM students
WHERE name LIKE '%a%a%';
""",
level="Medium",
follow_up="Ask: Why is 'Aarav' returned even though the first letter is a capital A?  Expect: SQLite LIKE ignores case for English letters.",
look_for="Chains wildcards correctly: '%a%a%'.")

❓ Q25  🟢 Easy
Find all students whose name starts with 'A'.

✅ MODEL ANSWER
SELECT name
FROM students
WHERE name LIKE 'A%';

📊 EXPECTED OUTPUT


,name
0,Aarav
1,Ananya
2,Arjun


👀 LOOK FOR: Pattern 'A%' (not '%A%').

❓ Q26  🟢 Easy
Find students whose name has exactly 6 letters.

✅ MODEL ANSWER
SELECT name
FROM students
WHERE name LIKE '______';

📊 EXPECTED OUTPUT


,name
0,Ananya
1,Vikram
2,Suresh
3,Nikhil


🔁 FOLLOW-UP: Ask: Is there a cleaner way than six underscores?  Expect: WHERE LENGTH(name) = 6.
👀 LOOK FOR: Knows _ matches exactly one character. Bonus for the LENGTH alternative.

❓ Q27  🟡 Medium
Find students whose name contains the letter 'a' at least twice.

✅ MODEL ANSWER
SELECT name
FROM students
WHERE name LIKE '%a%a%';

📊 EXPECTED OUTPUT


,name
0,Aarav
1,Ananya


🔁 FOLLOW-UP: Ask: Why is 'Aarav' returned even though the first letter is a capital A?  Expect: SQLite LIKE ignores case for English letters.
👀 LOOK FOR: Chains wildcards correctly: '%a%a%'.



---
# 🔟 Aggregate Functions
**What you are testing:** COUNT/SUM/AVG/MIN/MAX, NULL handling and conditional aggregation.

### 🗣️ Concept question
**Q: What is the difference between `COUNT(*)`, `COUNT(col)` and `COUNT(DISTINCT col)`?**
> **Expected answer:** `COUNT(*)` counts rows. `COUNT(col)` counts non-NULL values. `COUNT(DISTINCT col)` counts unique non-NULL values. Aggregates other than `COUNT(*)` ignore NULLs.

In [ ]:
ask("Give me a quick summary of the marks table: number of entries, average, highest and lowest mark.",
"""
SELECT COUNT(*)            AS entries,
       ROUND(AVG(marks),2) AS average_mark,
       MAX(marks)          AS highest,
       MIN(marks)          AS lowest
FROM marks;
""",
level="Easy",
look_for="Uses several aggregates in one SELECT. Rounds sensibly.")

ask("How many students are there in total, how many have a department, and how many different departments are used?",
"""
SELECT COUNT(*)                AS all_students,
       COUNT(dept_id)          AS students_with_department,
       COUNT(DISTINCT dept_id) AS departments_in_use
FROM students;
""",
level="Medium",
follow_up="Ask: Why is the second number smaller than the first?  Expect: COUNT(col) skips NULLs (Pooja has no department).",
look_for="Predicts 13, 12, 3 before running it.")

ask("What percentage of mark entries are below 50?",
"""
SELECT ROUND(100.0 * SUM(CASE WHEN marks < 50 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_below_50
FROM marks;
""",
level="Hard",
follow_up="Ask: Why 100.0 and not 100?  Expect: integer division in SQLite would truncate the result to 0.",
look_for="Conditional aggregation (SUM + CASE). Knows about integer division.")

❓ Q28  🟢 Easy
Give me a quick summary of the marks table: number of entries, average, highest and lowest mark.

✅ MODEL ANSWER
SELECT COUNT(*)            AS entries,
       ROUND(AVG(marks),2) AS average_mark,
       MAX(marks)          AS highest,
       MIN(marks)          AS lowest
FROM marks;

📊 EXPECTED OUTPUT


,entries,average_mark,highest,lowest
0,20,74.1,95,39


👀 LOOK FOR: Uses several aggregates in one SELECT. Rounds sensibly.

❓ Q29  🟡 Medium
How many students are there in total, how many have a department, and how many different departments are used?

✅ MODEL ANSWER
SELECT COUNT(*)                AS all_students,
       COUNT(dept_id)          AS students_with_department,
       COUNT(DISTINCT dept_id) AS departments_in_use
FROM students;

📊 EXPECTED OUTPUT


,all_students,students_with_department,departments_in_use
0,13,12,3


🔁 FOLLOW-UP: Ask: Why is the second number smaller than the first?  Expect: COUNT(col) skips NULLs (Pooja has no department).
👀 LOOK FOR: Predicts 13, 12, 3 before running it.

❓ Q30  🔴 Hard
What percentage of mark entries are below 50?

✅ MODEL ANSWER
SELECT ROUND(100.0 * SUM(CASE WHEN marks < 50 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_below_50
FROM marks;

📊 EXPECTED OUTPUT


,pct_below_50
0,10.0


🔁 FOLLOW-UP: Ask: Why 100.0 and not 100?  Expect: integer division in SQLite would truncate the result to 0.
👀 LOOK FOR: Conditional aggregation (SUM + CASE). Knows about integer division.



---
# 1️⃣1️⃣ GROUP BY
**What you are testing:** grouping, joining before grouping, pivoting with `CASE`.

### 🗣️ Concept question
**Q: Why can't you `SELECT name, COUNT(*) FROM students GROUP BY city`?**
> **Expected answer:** `name` is neither in `GROUP BY` nor inside an aggregate, so it is ambiguous: each city group has many names. Every selected column must be grouped or aggregated. (SQLite allows it and returns an arbitrary row, but standard SQL and most databases reject it.)

In [ ]:
ask("What is the average mark for each subject? Show the subject NAME, best average first.",
"""
SELECT sub.subject_name,
       ROUND(AVG(m.marks), 2) AS avg_marks
FROM marks m
JOIN subjects sub ON sub.subject_id = m.subject_id
GROUP BY sub.subject_id, sub.subject_name
ORDER BY avg_marks DESC;
""",
level="Medium",
look_for="JOIN + GROUP BY + ORDER BY on an alias.")

ask("How many students come from each city? Show the biggest city first.",
"""
SELECT city, COUNT(*) AS student_count
FROM students
GROUP BY city
ORDER BY student_count DESC, city;
""",
level="Easy",
look_for="Adds a tie-breaker in ORDER BY (nice touch).")

ask("For each department, show how many male and how many female students there are, side by side.",
"""
SELECT dept_id,
       SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) AS male_students,
       SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female_students
FROM students
GROUP BY dept_id;
""",
level="Hard",
follow_up="Ask: What is this technique called?  Expect: conditional aggregation, or 'pivoting' rows into columns.",
look_for="Does not need a separate query per gender.")

❓ Q31  🟡 Medium
What is the average mark for each subject? Show the subject NAME, best average first.

✅ MODEL ANSWER
SELECT sub.subject_name,
       ROUND(AVG(m.marks), 2) AS avg_marks
FROM marks m
JOIN subjects sub ON sub.subject_id = m.subject_id
GROUP BY sub.subject_id, sub.subject_name
ORDER BY avg_marks DESC;

📊 EXPECTED OUTPUT


,subject_name,avg_marks
0,Data Structures,83.25
1,Signals,79.50
2,Digital Circuits,78.50
3,Database Systems,75.00
4,Thermodynamics,47.33


👀 LOOK FOR: JOIN + GROUP BY + ORDER BY on an alias.

❓ Q32  🟢 Easy
How many students come from each city? Show the biggest city first.

✅ MODEL ANSWER
SELECT city, COUNT(*) AS student_count
FROM students
GROUP BY city
ORDER BY student_count DESC, city;

📊 EXPECTED OUTPUT


,city,student_count
0,Hyderabad,4
1,Chennai,3
2,Bengaluru,2
3,Mumbai,2
4,Pune,2


👀 LOOK FOR: Adds a tie-breaker in ORDER BY (nice touch).

❓ Q33  🔴 Hard
For each department, show how many male and how many female students there are, side by side.

✅ MODEL ANSWER
SELECT dept_id,
       SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) AS male_students,
       SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female_students
FROM students
GROUP BY dept_id;

📊 EXPECTED OUTPUT


,dept_id,male_students,female_students
0,NaN,0,1
1,1.0,3,2
2,2.0,1,3
3,3.0,3,0


🔁 FOLLOW-UP: Ask: What is this technique called?  Expect: conditional aggregation, or 'pivoting' rows into columns.
👀 LOOK FOR: Does not need a separate query per gender.



---
# 1️⃣2️⃣ HAVING
**What you are testing:** filtering groups, and `WHERE` vs `HAVING`.

### 🗣️ Concept question
**Q: What is the difference between `WHERE` and `HAVING`?**
> **Expected answer:** `WHERE` filters **rows before** grouping and cannot use aggregates. `HAVING` filters **groups after** grouping and can use aggregates. Logical order: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT.

In [ ]:
ask("Which cities have more than 2 students?",
"""
SELECT city, COUNT(*) AS student_count
FROM students
GROUP BY city
HAVING COUNT(*) > 2;
""",
level="Easy",
follow_up="Ask: Could you put COUNT(*) > 2 in WHERE?  Expect: no, aggregates are not allowed in WHERE.",
look_for="HAVING with an aggregate, not WHERE.")

ask("Find students who have at least 2 marks AND an average of 80 or more.",
"""
SELECT s.name,
       COUNT(*)            AS subjects_taken,
       ROUND(AVG(m.marks),1) AS avg_marks
FROM students s
JOIN marks m ON m.student_id = s.student_id
GROUP BY s.student_id, s.name
HAVING COUNT(*) >= 2 AND AVG(m.marks) >= 80
ORDER BY avg_marks DESC;
""",
level="Medium",
look_for="Two aggregate conditions in one HAVING. Groups by student_id, not just name (two students could share a name).")

ask("Which single subject has the highest average mark?",
"""
SELECT sub.subject_name, ROUND(AVG(m.marks),2) AS avg_marks
FROM marks m
JOIN subjects sub ON sub.subject_id = m.subject_id
GROUP BY sub.subject_id, sub.subject_name
ORDER BY avg_marks DESC
LIMIT 1;
""",
level="Medium",
follow_up="Ask: What if two subjects tie for the highest average?  Expect: LIMIT 1 hides the tie; use HAVING AVG(...) = (SELECT MAX(...) ...) or RANK().",
look_for="Considers ties.")

❓ Q34  🟢 Easy
Which cities have more than 2 students?

✅ MODEL ANSWER
SELECT city, COUNT(*) AS student_count
FROM students
GROUP BY city
HAVING COUNT(*) > 2;

📊 EXPECTED OUTPUT


,city,student_count
0,Chennai,3
1,Hyderabad,4


🔁 FOLLOW-UP: Ask: Could you put COUNT(*) > 2 in WHERE?  Expect: no, aggregates are not allowed in WHERE.
👀 LOOK FOR: HAVING with an aggregate, not WHERE.

❓ Q35  🟡 Medium
Find students who have at least 2 marks AND an average of 80 or more.

✅ MODEL ANSWER
SELECT s.name,
       COUNT(*)            AS subjects_taken,
       ROUND(AVG(m.marks),1) AS avg_marks
FROM students s
JOIN marks m ON m.student_id = s.student_id
GROUP BY s.student_id, s.name
HAVING COUNT(*) >= 2 AND AVG(m.marks) >= 80
ORDER BY avg_marks DESC;

📊 EXPECTED OUTPUT


,name,subjects_taken,avg_marks
0,Priya,2,90.0
1,Sneha,2,85.5
2,Ananya,2,85.5
3,Aarav,2,81.5
4,Meera,2,81.0
5,Divya,2,80.0


👀 LOOK FOR: Two aggregate conditions in one HAVING. Groups by student_id, not just name (two students could share a name).

❓ Q36  🟡 Medium
Which single subject has the highest average mark?

✅ MODEL ANSWER
SELECT sub.subject_name, ROUND(AVG(m.marks),2) AS avg_marks
FROM marks m
JOIN subjects sub ON sub.subject_id = m.subject_id
GROUP BY sub.subject_id, sub.subject_name
ORDER BY avg_marks DESC
LIMIT 1;

📊 EXPECTED OUTPUT


,subject_name,avg_marks
0,Data Structures,83.25


🔁 FOLLOW-UP: Ask: What if two subjects tie for the highest average?  Expect: LIMIT 1 hides the tie; use HAVING AVG(...) = (SELECT MAX(...) ...) or RANK().
👀 LOOK FOR: Considers ties.



---
# 1️⃣3️⃣ JOINs
**What you are testing:** the most important topic in most interviews: join types, NULL handling, multi-table joins.

### 🗣️ Concept questions
**Q1: What is the difference between `INNER JOIN` and `LEFT JOIN`?**
> **Expected answer:** `INNER JOIN` keeps only rows that match in both tables. `LEFT JOIN` keeps **all** rows from the left table and fills the right side with NULLs when there is no match.

**Q2: With a `LEFT JOIN`, does a filter on the right table go in `ON` or `WHERE`?**
> **Expected answer:** A condition on the right table in `WHERE` (e.g. `d.hod = 'Dr. Rao'`) turns the LEFT JOIN into an INNER JOIN because NULL rows get filtered out. Put it in `ON` to keep unmatched left rows. The exception is `WHERE right.col IS NULL`, used on purpose to find non-matches.

In [ ]:
ask("Show each student's name together with their department name.",
"""
SELECT s.name, d.dept_name
FROM students s
INNER JOIN departments d ON d.dept_id = s.dept_id;
""",
level="Easy",
follow_up="Ask: How many rows come back, and who is missing?  Expect: 12 rows; Pooja is missing because she has no department.",
look_for="Correct ON condition. Notices the missing student.")

ask("Show ALL students with their department name. If there is no department, show 'Unassigned'.",
"""
SELECT s.name,
       COALESCE(d.dept_name, 'Unassigned') AS department
FROM students s
LEFT JOIN departments d ON d.dept_id = s.dept_id;
""",
level="Medium",
look_for="LEFT JOIN + COALESCE (or IFNULL).")

ask("Which departments currently have no students?",
"""
SELECT d.dept_name
FROM departments d
LEFT JOIN students s ON s.dept_id = d.dept_id
WHERE s.student_id IS NULL;
""",
level="Medium",
follow_up="Ask: Another way to write it?  Expect: NOT EXISTS or NOT IN (with the NULL caveat).",
look_for="The anti-join pattern: LEFT JOIN ... WHERE right_key IS NULL.")

ask("Show the average mark per department, including departments that have no students or no marks.",
"""
SELECT d.dept_name,
       ROUND(AVG(m.marks), 2) AS avg_marks
FROM departments d
LEFT JOIN students s ON s.dept_id    = d.dept_id
LEFT JOIN marks m    ON m.student_id = s.student_id
GROUP BY d.dept_id, d.dept_name
ORDER BY avg_marks DESC;
""",
level="Hard",
follow_up="Ask: Why do you need LEFT JOIN twice?  Expect: an INNER JOIN on the second join would drop Civil again.",
look_for="Starts from departments (the table that must be fully kept) and chains LEFT JOINs.")

ask("List every pair of students who live in the same city (each pair once).",
"""
SELECT a.name AS student_1, b.name AS student_2, a.city
FROM students a
JOIN students b
  ON a.city = b.city
 AND a.student_id < b.student_id
ORDER BY a.city, student_1;
""",
level="Hard",
follow_up="Ask: Why a.student_id < b.student_id?  Expect: it removes pairs of a student with themselves and mirrored duplicates (A,B) and (B,A).",
look_for="Self join with an inequality condition.")


❓ Q37  🟢 Easy
Show each student's name together with their department name.

✅ MODEL ANSWER
SELECT s.name, d.dept_name
FROM students s
INNER JOIN departments d ON d.dept_id = s.dept_id;

📊 EXPECTED OUTPUT


,name,dept_name
0,Aarav,Computer Science
1,Priya,Computer Science
2,Rohan,Electronics
3,Sneha,Electronics
4,Karthik,Mechanical
5,Ananya,Computer Science
6,Vikram,Mechanical
7,Meera,Electronics
8,Arjun,Computer Science
9,Divya,Electronics


🔁 FOLLOW-UP: Ask: How many rows come back, and who is missing?  Expect: 12 rows; Pooja is missing because she has no department.
👀 LOOK FOR: Correct ON condition. Notices the missing student.

❓ Q38  🟡 Medium
Show ALL students with their department name. If there is no department, show 'Unassigned'.

✅ MODEL ANSWER
SELECT s.name,
       COALESCE(d.dept_name, 'Unassigned') AS department
FROM students s
LEFT JOIN departments d ON d.dept_id = s.dept_id;

📊 EXPECTED OUTPUT


,name,department
0,Aarav,Computer Science
1,Priya,Computer Science
2,Rohan,Electronics
3,Sneha,Electronics
4,Karthik,Mechanical
5,Ananya,Computer Science
6,Vikram,Mechanical
7,Meera,Electronics
8,Arjun,Computer Science
9,Divya,Electronics


👀 LOOK FOR: LEFT JOIN + COALESCE (or IFNULL).

❓ Q39  🟡 Medium
Which departments currently have no students?

✅ MODEL ANSWER
SELECT d.dept_name
FROM departments d
LEFT JOIN students s ON s.dept_id = d.dept_id
WHERE s.student_id IS NULL;

📊 EXPECTED OUTPUT


...

🔁 FOLLOW-UP: Ask: Another way to write it?  Expect: NOT EXISTS or NOT IN (with the NULL caveat).
👀 LOOK FOR: The anti-join pattern: LEFT JOIN ... WHERE right_key IS NULL.

❓ Q40  🔴 Hard
Show the average mark per department, including departments that have no students or no marks.

✅ MODEL ANSWER
SELECT d.dept_name,
       ROUND(AVG(m.marks), 2) AS avg_marks
FROM departments d
LEFT JOIN students s ON s.dept_id    = d.dept_id
LEFT JOIN marks m    ON m.student_id = s.student_id
GROUP BY d.dept_id, d.dept_name
ORDER BY avg_marks DESC;

📊 EXPECTED OUTPUT


,dept_name,avg_marks
0,Computer Science,81.25
1,Electronics,79.00
2,Mechanical,47.33
3,Civil,NaN


🔁 FOLLOW-UP: Ask: Why do you need LEFT JOIN twice?  Expect: an INNER JOIN on the second join would drop Civil again.
👀 LOOK FOR: Starts from departments (the table that must be fully kept) and chains LEFT JOINs.

❓ Q41  🔴 Hard
List every pair of students who live in the same city (each pair once).

✅ MODEL ANSWER
SELECT a.name AS student_1, b.name AS student_2, a.city
FROM students a
JOIN students b
  ON a.city = b.city
 AND a.student_id < b.student_id
ORDER BY a.city, student_1;

📊 EXPECTED OUTPUT


,student_1,student_2,city
0,Rohan,Ananya,Bengaluru
1,Karthik,Divya,Chennai
2,Priya,Divya,Chennai
3,Priya,Karthik,Chennai
4,Aarav,Meera,Hyderabad
5,Aarav,Nikhil,Hyderabad
6,Aarav,Sneha,Hyderabad
7,Meera,Nikhil,Hyderabad
8,Sneha,Meera,Hyderabad
9,Sneha,Nikhil,Hyderabad


🔁 FOLLOW-UP: Ask: Why a.student_id < b.student_id?  Expect: it removes pairs of a student with themselves and mirrored duplicates (A,B) and (B,A).
👀 LOOK FOR: Self join with an inequality condition.



---
# 1️⃣4️⃣ Subqueries
**What you are testing:** scalar, list and correlated subqueries, `EXISTS`.

### 🗣️ Concept question
**Q: What is a correlated subquery, and how does it differ from a normal one?**
> **Expected answer:** A normal subquery runs once and is independent of the outer query. A correlated subquery references a column of the outer query, so it is logically re-evaluated for every outer row. It is more powerful but can be slower.

In [ ]:
ask("Show the marks that are above the overall average mark, with the student's name.",
"""
SELECT s.name, m.marks
FROM marks m
JOIN students s ON s.student_id = m.student_id
WHERE m.marks > (SELECT AVG(marks) FROM marks)
ORDER BY m.marks DESC;
""",
level="Easy",
look_for="Scalar subquery in WHERE.")

ask("Which students have no marks recorded? Use NOT EXISTS.",
"""
SELECT s.name
FROM students s
WHERE NOT EXISTS (SELECT 1
                  FROM marks m
                  WHERE m.student_id = s.student_id);
""",
level="Medium",
follow_up="Ask: Why prefer NOT EXISTS over NOT IN?  Expect: NOT IN breaks (returns nothing) if the subquery can return NULL.",
look_for="Correlated subquery. Understands that SELECT 1 is just a placeholder.")

ask("Find the second highest mark WITHOUT using LIMIT or window functions.",
"""
SELECT MAX(marks) AS second_highest_mark
FROM marks
WHERE marks < (SELECT MAX(marks) FROM marks);
""",
level="Medium",
follow_up="Ask: How would you generalise to the Nth highest?  Expect: window function DENSE_RANK, or LIMIT 1 OFFSET N-1 on DISTINCT values.",
look_for="Nested MAX. A classic that nearly every company asks.")

ask("For each subject, list the students who scored ABOVE THAT SUBJECT's average.",
"""
SELECT sub.subject_name, s.name, m.marks
FROM marks m
JOIN students s   ON s.student_id   = m.student_id
JOIN subjects sub ON sub.subject_id = m.subject_id
WHERE m.marks > (SELECT AVG(m2.marks)
                 FROM marks m2
                 WHERE m2.subject_id = m.subject_id)
ORDER BY sub.subject_name, m.marks DESC;
""",
level="Hard",
follow_up="Ask: How would you do it without a correlated subquery?  Expect: a CTE or window function (AVG() OVER (PARTITION BY subject_id)).",
look_for="Correlated subquery referencing the outer alias m.")


❓ Q42  🟢 Easy
Show the marks that are above the overall average mark, with the student's name.

✅ MODEL ANSWER
SELECT s.name, m.marks
FROM marks m
JOIN students s ON s.student_id = m.student_id
WHERE m.marks > (SELECT AVG(marks) FROM marks)
ORDER BY m.marks DESC;

📊 EXPECTED OUTPUT


,name,marks
0,Ananya,95
1,Priya,92
2,Sneha,90
3,Priya,88
4,Divya,86
5,Aarav,85
6,Meera,83
7,Sneha,81
8,Meera,79
9,Aarav,78


👀 LOOK FOR: Scalar subquery in WHERE.

❓ Q43  🟡 Medium
Which students have no marks recorded? Use NOT EXISTS.

✅ MODEL ANSWER
SELECT s.name
FROM students s
WHERE NOT EXISTS (SELECT 1
                  FROM marks m
                  WHERE m.student_id = s.student_id);

📊 EXPECTED OUTPUT


,name
0,Nikhil


🔁 FOLLOW-UP: Ask: Why prefer NOT EXISTS over NOT IN?  Expect: NOT IN breaks (returns nothing) if the subquery can return NULL.
👀 LOOK FOR: Correlated subquery. Understands that SELECT 1 is just a placeholder.

❓ Q44  🟡 Medium
Find the second highest mark WITHOUT using LIMIT or window functions.

✅ MODEL ANSWER
SELECT MAX(marks) AS second_highest_mark
FROM marks
WHERE marks < (SELECT MAX(marks) FROM marks);

📊 EXPECTED OUTPUT


,second_highest_mark
0,92


🔁 FOLLOW-UP: Ask: How would you generalise to the Nth highest?  Expect: window function DENSE_RANK, or LIMIT 1 OFFSET N-1 on DISTINCT values.
👀 LOOK FOR: Nested MAX. A classic that nearly every company asks.

❓ Q45  🔴 Hard
For each subject, list the students who scored ABOVE THAT SUBJECT's average.

✅ MODEL ANSWER
SELECT sub.subject_name, s.name, m.marks
FROM marks m
JOIN students s   ON s.student_id   = m.student_id
JOIN subjects sub ON sub.subject_id = m.subject_id
WHERE m.marks > (SELECT AVG(m2.marks)
                 FROM marks m2
                 WHERE m2.subject_id = m.subject_id)
ORDER BY sub.subject_name, m.marks DESC;

📊 EXPECTED OUTPUT


,subject_name,name,marks
0,Data Structures,Ananya,95
1,Data Structures,Priya,88
2,Database Systems,Priya,92
3,Database Systems,Aarav,85
4,Database Systems,Ananya,76
5,Digital Circuits,Sneha,90
6,Digital Circuits,Meera,83
7,Signals,Divya,86
8,Signals,Sneha,81
9,Thermodynamics,Karthik,55


🔁 FOLLOW-UP: Ask: How would you do it without a correlated subquery?  Expect: a CTE or window function (AVG() OVER (PARTITION BY subject_id)).
👀 LOOK FOR: Correlated subquery referencing the outer alias m.



---
# 1️⃣5️⃣ CTEs
**What you are testing:** readable multi-step queries, chaining, recursion.

### 🗣️ Concept question
**Q: When would you use a CTE instead of a subquery?**
> **Expected answer:** When the query has several steps, when the same result is used more than once, or for readability. CTEs can be chained, and only CTEs support recursion (`WITH RECURSIVE`). Performance is usually the same as an inline subquery.

In [ ]:
ask("Using a CTE, list students whose average mark is 75 or more.",
"""
WITH student_avg AS (
    SELECT student_id, ROUND(AVG(marks), 1) AS avg_marks
    FROM marks
    GROUP BY student_id
)
SELECT s.name, sa.avg_marks
FROM student_avg sa
JOIN students s ON s.student_id = sa.student_id
WHERE sa.avg_marks >= 75
ORDER BY sa.avg_marks DESC;
""",
level="Medium",
look_for="Defines the CTE first, then queries it like a table.")

ask("Using two chained CTEs, find the average of student averages for each department.",
"""
WITH student_avg AS (
    SELECT student_id, AVG(marks) AS avg_marks
    FROM marks
    GROUP BY student_id
),
dept_avg AS (
    SELECT s.dept_id, ROUND(AVG(sa.avg_marks), 2) AS dept_avg_marks
    FROM student_avg sa
    JOIN students s ON s.student_id = sa.student_id
    GROUP BY s.dept_id
)
SELECT d.dept_name, da.dept_avg_marks
FROM dept_avg da
JOIN departments d ON d.dept_id = da.dept_id
ORDER BY da.dept_avg_marks DESC;
""",
level="Hard",
follow_up="Ask: Why is this different from the plain average of all marks in a department?  Expect: it gives every student equal weight, regardless of how many subjects they took.",
look_for="Second CTE can read the first. Comfortable with multi-step logic.")

ask("Without using any table, generate the numbers 1 to 5. (Hint: recursive CTE.)",
"""
WITH RECURSIVE numbers(n) AS (
    SELECT 1                         -- anchor member
    UNION ALL
    SELECT n + 1 FROM numbers WHERE n < 5   -- recursive member
)
SELECT n FROM numbers;
""",
level="Hard",
follow_up="Ask: What happens if you forget the WHERE n < 5?  Expect: infinite recursion.",
look_for="Knows the anchor + recursive + stop-condition structure.")


❓ Q46  🟡 Medium
Using a CTE, list students whose average mark is 75 or more.

✅ MODEL ANSWER
WITH student_avg AS (
    SELECT student_id, ROUND(AVG(marks), 1) AS avg_marks
    FROM marks
    GROUP BY student_id
)
SELECT s.name, sa.avg_marks
FROM student_avg sa
JOIN students s ON s.student_id = sa.student_id
WHERE sa.avg_marks >= 75
ORDER BY sa.avg_marks DESC;

📊 EXPECTED OUTPUT


,name,avg_marks
0,Priya,90.0
1,Sneha,85.5
2,Ananya,85.5
3,Aarav,81.5
4,Meera,81.0
5,Divya,80.0


👀 LOOK FOR: Defines the CTE first, then queries it like a table.

❓ Q47  🔴 Hard
Using two chained CTEs, find the average of student averages for each department.

✅ MODEL ANSWER
WITH student_avg AS (
    SELECT student_id, AVG(marks) AS avg_marks
    FROM marks
    GROUP BY student_id
),
dept_avg AS (
    SELECT s.dept_id, ROUND(AVG(sa.avg_marks), 2) AS dept_avg_marks
    FROM student_avg sa
    JOIN students s ON s.student_id = sa.student_id
    GROUP BY s.dept_id
)
SELECT d.dept_name, da.dept_avg_marks
FROM dept_avg da
JOIN departments d ON d.dept_id = da.dept_id
ORDER BY da.dept_avg_marks DESC;

📊 EXPECTED OUTPUT


,dept_name,dept_avg_marks
0,Computer Science,81.25
1,Electronics,79.00
2,Mechanical,47.33


🔁 FOLLOW-UP: Ask: Why is this different from the plain average of all marks in a department?  Expect: it gives every student equal weight, regardless of how many subjects they took.
👀 LOOK FOR: Second CTE can read the first. Comfortable with multi-step logic.

❓ Q48  🔴 Hard
Without using any table, generate the numbers 1 to 5. (Hint: recursive CTE.)

✅ MODEL ANSWER
WITH RECURSIVE numbers(n) AS (
    SELECT 1                         -- anchor member
    UNION ALL
    SELECT n + 1 FROM numbers WHERE n < 5   -- recursive member
)
SELECT n FROM numbers;

📊 EXPECTED OUTPUT


,n
0,1
1,2
2,3
3,4
4,5


🔁 FOLLOW-UP: Ask: What happens if you forget the WHERE n < 5?  Expect: infinite recursion.
👀 LOOK FOR: Knows the anchor + recursive + stop-condition structure.



---
# 1️⃣6️⃣ Window Functions
**What you are testing:** ranking, running totals, comparing a row with its group, `LAG`.

### 🗣️ Concept questions
**Q1: How is a window function different from `GROUP BY`?**
> **Expected answer:** `GROUP BY` collapses rows into one row per group. A window function calculates over a group of rows but **keeps every row**.

**Q2: Explain `ROW_NUMBER`, `RANK` and `DENSE_RANK` using the scores 90, 80, 80, 70.**
> **Expected answer:** ROW_NUMBER → 1, 2, 3, 4 (always unique). RANK → 1, 2, 2, 4 (gap after the tie). DENSE_RANK → 1, 2, 2, 3 (no gap).

**Q3: Can you filter on a window function in `WHERE`?**
> **Expected answer:** No, window functions are evaluated after `WHERE`. Wrap the query in a CTE or subquery and filter there.

In [ ]:
ask("Show the first 16 marks from highest to lowest, with ROW_NUMBER, RANK and DENSE_RANK side by side.",
"""
SELECT s.name, m.marks,
       ROW_NUMBER() OVER (ORDER BY m.marks DESC, m.mark_id) AS row_num,
       RANK()       OVER (ORDER BY m.marks DESC)            AS rank_no,
       DENSE_RANK() OVER (ORDER BY m.marks DESC)            AS dense_rank_no
FROM marks m
JOIN students s ON s.student_id = m.student_id
ORDER BY m.marks DESC, m.mark_id
LIMIT 16;
""",
level="Medium",
follow_up="Point at the two 72s. Ask: What does each function give for the next mark (67)?  Expect: ROW_NUMBER 15, RANK 15, DENSE_RANK 14.",
look_for="Explains the tie behaviour in their own words.")

ask("Rank students by their TOTAL marks, highest total first. Ties must share a rank and there must be no gaps.",
"""
WITH totals AS (
    SELECT student_id, SUM(marks) AS total_marks
    FROM marks
    GROUP BY student_id
)
SELECT s.name, t.total_marks,
       DENSE_RANK() OVER (ORDER BY t.total_marks DESC) AS overall_rank
FROM totals t
JOIN students s ON s.student_id = t.student_id
ORDER BY overall_rank, s.name;
""",
level="Medium",
look_for="Aggregates first (CTE), then ranks. Chooses DENSE_RANK because of 'no gaps'.")

ask("Show the top 2 scorers in every subject.",
"""
WITH ranked AS (
    SELECT m.subject_id, m.student_id, m.marks,
           ROW_NUMBER() OVER (PARTITION BY m.subject_id ORDER BY m.marks DESC) AS rn
    FROM marks m
)
SELECT sub.subject_name, s.name, r.marks, r.rn
FROM ranked r
JOIN students s   ON s.student_id   = r.student_id
JOIN subjects sub ON sub.subject_id = r.subject_id
WHERE r.rn <= 2
ORDER BY sub.subject_name, r.rn;
""",
level="Hard",
follow_up="Ask: What if there is a tie for 2nd place and we want to include both?  Expect: switch ROW_NUMBER to RANK or DENSE_RANK.",
look_for="PARTITION BY + CTE + filter on rank. The classic 'top N per group' pattern.")

ask("For students 1 to 4, show how each mark compares with the PREVIOUS subject's mark.",
"""
SELECT student_id, subject_id, marks,
       LAG(marks) OVER (PARTITION BY student_id ORDER BY subject_id) AS previous_mark,
       marks - LAG(marks) OVER (PARTITION BY student_id ORDER BY subject_id) AS change
FROM marks
WHERE student_id IN (1, 2, 3, 4)
ORDER BY student_id, subject_id;
""",
level="Hard",
follow_up="Ask: Why is previous_mark NULL on some rows?  Expect: the first row of each partition has no previous row. LAG(marks, 1, 0) would supply a default.",
look_for="Knows LAG/LEAD and why PARTITION BY matters.")

ask("Show a running total of marks for each student (students 1 to 4), in subject order.",
"""
SELECT student_id, subject_id, marks,
       SUM(marks) OVER (
           PARTITION BY student_id
           ORDER BY subject_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM marks
WHERE student_id IN (1, 2, 3, 4)
ORDER BY student_id, subject_id;
""",
level="Hard",
look_for="SUM() OVER with ORDER BY. Bonus if they know the frame clause.")

ask("For students 1 to 3, show each mark as a percentage of that student's total marks.",
"""
SELECT student_id, subject_id, marks,
       SUM(marks) OVER (PARTITION BY student_id) AS student_total,
       ROUND(100.0 * marks / SUM(marks) OVER (PARTITION BY student_id), 1) AS pct_of_total
FROM marks
WHERE student_id IN (1, 2, 3)
ORDER BY student_id, subject_id;
""",
level="Hard",
look_for="Uses an aggregate window to avoid a self-join or correlated subquery.")


❓ Q49  🟡 Medium
Show the first 16 marks from highest to lowest, with ROW_NUMBER, RANK and DENSE_RANK side by side.

✅ MODEL ANSWER
SELECT s.name, m.marks,
       ROW_NUMBER() OVER (ORDER BY m.marks DESC, m.mark_id) AS row_num,
       RANK()       OVER (ORDER BY m.marks DESC)            AS rank_no,
       DENSE_RANK() OVER (ORDER BY m.marks DESC)            AS dense_rank_no
FROM marks m
JOIN students s ON s.student_id = m.student_id
ORDER BY m.marks DESC, m.mark_id
LIMIT 16;

📊 EXPECTED OUTPUT


,name,marks,row_num,rank_no,dense_rank_no
0,Ananya,95,1,1,1
1,Priya,92,2,2,2
2,Sneha,90,3,3,3
3,Priya,88,4,4,4
4,Divya,86,5,5,5
5,Aarav,85,6,6,6
6,Meera,83,7,7,7
7,Sneha,81,8,8,8
8,Meera,79,9,9,9
9,Aarav,78,10,10,10


🔁 FOLLOW-UP: Point at the two 72s. Ask: What does each function give for the next mark (67)?  Expect: ROW_NUMBER 15, RANK 15, DENSE_RANK 14.
👀 LOOK FOR: Explains the tie behaviour in their own words.

❓ Q50  🟡 Medium
Rank students by their TOTAL marks, highest total first. Ties must share a rank and there must be no gaps.

✅ MODEL ANSWER
WITH totals AS (
    SELECT student_id, SUM(marks) AS total_marks
    FROM marks
    GROUP BY student_id
)
SELECT s.name, t.total_marks,
       DENSE_RANK() OVER (ORDER BY t.total_marks DESC) AS overall_rank
FROM totals t
JOIN students s ON s.student_id = t.student_id
ORDER BY overall_rank, s.name;

📊 EXPECTED OUTPUT


,name,total_marks,overall_rank
0,Priya,180,1
1,Ananya,171,2
2,Sneha,171,2
3,Aarav,163,3
4,Meera,162,4
5,Divya,160,5
6,Rohan,139,6
7,Arjun,136,7
8,Pooja,58,8
9,Karthik,55,9


👀 LOOK FOR: Aggregates first (CTE), then ranks. Chooses DENSE_RANK because of 'no gaps'.

❓ Q51  🔴 Hard
Show the top 2 scorers in every subject.

✅ MODEL ANSWER
WITH ranked AS (
    SELECT m.subject_id, m.student_id, m.marks,
           ROW_NUMBER() OVER (PARTITION BY m.subject_id ORDER BY m.marks DESC) AS rn
    FROM marks m
)
SELECT sub.subject_name, s.name, r.marks, r.rn
FROM ranked r
JOIN students s   ON s.student_id   = r.student_id
JOIN subjects sub ON sub.subject_id = r.subject_id
WHERE r.rn <= 2
ORDER BY sub.subject_name, r.rn;

📊 EXPECTED OUTPUT


...

🔁 FOLLOW-UP: Ask: Why is previous_mark NULL on some rows?  Expect: the first row of each partition has no previous row. LAG(marks, 1, 0) would supply a default.
👀 LOOK FOR: Knows LAG/LEAD and why PARTITION BY matters.

❓ Q53  🔴 Hard
Show a running total of marks for each student (students 1 to 4), in subject order.

✅ MODEL ANSWER
SELECT student_id, subject_id, marks,
       SUM(marks) OVER (
           PARTITION BY student_id
           ORDER BY subject_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM marks
WHERE student_id IN (1, 2, 3, 4)
ORDER BY student_id, subject_id;

📊 EXPECTED OUTPUT


,student_id,subject_id,marks,running_total
0,1,101,85,85
1,1,102,78,163
2,2,101,92,92
3,2,102,88,180
4,3,103,67,67
5,3,104,72,139
6,4,103,90,90
7,4,104,81,171


👀 LOOK FOR: SUM() OVER with ORDER BY. Bonus if they know the frame clause.

❓ Q54  🔴 Hard
For students 1 to 3, show each mark as a percentage of that student's total marks.

✅ MODEL ANSWER
SELECT student_id, subject_id, marks,
       SUM(marks) OVER (PARTITION BY student_id) AS student_total,
       ROUND(100.0 * marks / SUM(marks) OVER (PARTITION BY student_id), 1) AS pct_of_total
FROM marks
WHERE student_id IN (1, 2, 3)
ORDER BY student_id, subject_id;

📊 EXPECTED OUTPUT


,student_id,subject_id,marks,student_total,pct_of_total
0,1,101,85,163,52.1
1,1,102,78,163,47.9
2,2,101,92,180,51.1
3,2,102,88,180,48.9
4,3,103,67,139,48.2
5,3,104,72,139,51.8


👀 LOOK FOR: Uses an aggregate window to avoid a self-join or correlated subquery.



---
# ⭐ Bonus: Classic "must-ask" questions
These come up in a large share of real interviews.

In [ ]:
ask("Find the 2nd highest total marks scored by a student. Make sure ties are handled correctly.",
"""
WITH totals AS (
    SELECT student_id, SUM(marks) AS total_marks
    FROM marks
    GROUP BY student_id
),
ranked AS (
    SELECT student_id, total_marks,
           DENSE_RANK() OVER (ORDER BY total_marks DESC) AS rnk
    FROM totals
)
SELECT s.name, r.total_marks
FROM ranked r
JOIN students s ON s.student_id = r.student_id
WHERE r.rnk = 2;
""",
level="Hard",
follow_up="Notice two students tie at 171. Ask: What would LIMIT 1 OFFSET 1 have returned?  Expect: only one of them, which hides the tie.",
look_for="Chooses DENSE_RANK over ROW_NUMBER.")

ask("Find the top-scoring student (by total marks) in each department.",
"""
WITH totals AS (
    SELECT s.student_id, s.name, s.dept_id, SUM(m.marks) AS total_marks
    FROM students s
    JOIN marks m ON m.student_id = s.student_id
    WHERE s.dept_id IS NOT NULL
    GROUP BY s.student_id, s.name, s.dept_id
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY dept_id ORDER BY total_marks DESC) AS rnk
    FROM totals
)
SELECT d.dept_name, r.name, r.total_marks
FROM ranked r
JOIN departments d ON d.dept_id = r.dept_id
WHERE r.rnk = 1
ORDER BY d.dept_name;
""",
level="Hard",
look_for="Aggregate, then rank per partition, then filter.")

ask("Find students whose average mark is above their own DEPARTMENT's average.",
"""
WITH student_avg AS (
    SELECT s.student_id, s.name, s.dept_id, AVG(m.marks) AS avg_marks
    FROM students s
    JOIN marks m ON m.student_id = s.student_id
    WHERE s.dept_id IS NOT NULL
    GROUP BY s.student_id, s.name, s.dept_id
),
with_dept_avg AS (
    SELECT name, dept_id, avg_marks,
           AVG(avg_marks) OVER (PARTITION BY dept_id) AS dept_avg
    FROM student_avg
)
SELECT name, dept_id,
       ROUND(avg_marks, 1) AS avg_marks,
       ROUND(dept_avg, 1)  AS dept_avg
FROM with_dept_avg
WHERE avg_marks > dept_avg
ORDER BY dept_id, avg_marks DESC;
""",
level="Hard",
follow_up="Ask: Why compute dept_avg in a CTE rather than directly in WHERE?  Expect: window functions cannot be used in WHERE.",
look_for="Window function inside a CTE, filtered in the outer query.")

ask("Find duplicate values: which cities appear more than once in the students table?",
"""
SELECT city, COUNT(*) AS times_appearing
FROM students
GROUP BY city
HAVING COUNT(*) > 1
ORDER BY times_appearing DESC, city;
""",
level="Easy",
look_for="GROUP BY + HAVING COUNT(*) > 1, the standard duplicate-finder.")

❓ Q55  🔴 Hard
Find the 2nd highest total marks scored by a student. Make sure ties are handled correctly.

✅ MODEL ANSWER
WITH totals AS (
    SELECT student_id, SUM(marks) AS total_marks
    FROM marks
    GROUP BY student_id
),
ranked AS (
    SELECT student_id, total_marks,
           DENSE_RANK() OVER (ORDER BY total_marks DESC) AS rnk
    FROM totals
)
SELECT s.name, r.total_marks
FROM ranked r
JOIN students s ON s.student_id = r.student_id
WHERE r.rnk = 2;

📊 EXPECTED OUTPUT


,name,total_marks
0,Sneha,171
1,Ananya,171


🔁 FOLLOW-UP:

 Notice two students tie at 171. Ask: What would LIMIT 1 OFFSET 1 have returned?  Expect: only one of them, which hides the tie.
👀 LOOK FOR: Chooses DENSE_RANK over ROW_NUMBER.

❓ Q56  🔴 Hard
Find the top-scoring student (by total marks) in each department.

✅ MODEL ANSWER
WITH totals AS (
    SELECT s.student_id, s.name, s.dept_id, SUM(m.marks) AS total_marks
    FROM students s
    JOIN marks m ON m.student_id = s.student_id
    WHERE s.dept_id IS NOT NULL
    GROUP BY s.student_id, s.name, s.dept_id
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY dept_id ORDER BY total_marks DESC) AS rnk
    FROM totals
)
SELECT d.dept_name, r.name, r.total_marks
FROM ranked r
JOIN departments d ON d.dept_id = r.dept_id
WHERE r.rnk = 1
ORDER BY d.dept_name;

📊 EXPECTED OUTPUT


,dept_name,name,total_marks
0,Computer Science,Priya,180
1,Electronics,Sneha,171
2,Mechanical,Karthik,55


👀 LOOK FOR: Aggregate, then rank per partition, then filter.

❓ Q57  🔴 Hard
Find students whose average mark is above their own DEPARTMENT's average.

✅ MODEL ANSWER
WITH student_avg AS (
    SELECT s.student_id, s.name, s.dept_id, AVG(m.marks) AS avg_marks
    FROM students s
    JOIN marks m ON m.student_id = s.student_id
    WHERE s.dept_id IS NOT NULL
    GROUP BY s.student_id, s.name, s.dept_id
),
with_dept_avg AS (
    SELECT name, dept_id, avg_marks,
           AVG(avg_marks) OVER (PARTITION BY dept_id) AS dept_avg
    FROM student_avg
)
SELECT name, dept_id,
       ROUND(avg_marks, 1) AS avg_marks,
       ROUND(dept_avg, 1)  AS dept_avg
FROM with_dept_avg
WHERE avg_marks > dept_avg
ORDER BY dept_id, avg_marks DESC;

📊 EXPECTED OUTPUT


,name,dept_id,avg_marks,dept_avg
0,Priya,1,90.0,81.3
1,Ananya,1,85.5,81.3
2,Aarav,1,81.5,81.3
3,Sneha,2,85.5,79.0
4,Meera,2,81.0,79.0
5,Divya,2,80.0,79.0
6,Karthik,3,55.0,47.3
7,Vikram,3,48.0,47.3


🔁 FOLLOW-UP: Ask: Why compute dept_avg in a CTE rather than directly in WHERE?  Expect: window functions cannot be used in WHERE.
👀 LOOK FOR: Window function inside a CTE, filtered in the outer query.

❓ Q58  🟢 Easy
Find duplicate values: which cities appear more than once in the students table?

✅ MODEL ANSWER
SELECT city, COUNT(*) AS times_appearing
FROM students
GROUP BY city
HAVING COUNT(*) > 1
ORDER BY times_appearing DESC, city;

📊 EXPECTED OUTPUT


,city,times_appearing
0,Hyderabad,4
1,Chennai,3
2,Bengaluru,2
3,Mumbai,2
4,Pune,2


👀 LOOK FOR: GROUP BY + HAVING COUNT(*) > 1, the standard duplicate-finder.



| # | Question | Expected answer |
|---|----------|-----------------|
| 1 | What is a primary key? | Uniquely identifies each row; cannot be NULL or duplicate. |
| 2 | What is a foreign key? | A column referencing another table's primary key; enforces referential integrity. |
| 3 | `DELETE` vs `TRUNCATE` vs `DROP`? | DELETE removes chosen rows (can filter, logged). TRUNCATE removes all rows quickly (no filter). DROP removes the whole table. |
| 4 | `UNION` vs `UNION ALL`? | UNION removes duplicates (slower); UNION ALL keeps them (faster). |
| 5 | What is an index and its trade-off? | A structure that speeds up reads on a column; slows writes and uses space. |
| 6 | What is normalisation? | Organising tables to reduce redundancy (1NF, 2NF, 3NF). |
| 7 | What does `COALESCE` do? | Returns the first non-NULL argument. |
| 8 | `CHAR` vs `VARCHAR`? | CHAR is fixed length (padded); VARCHAR is variable length. |
| 9 | What is a view? | A saved query that behaves like a virtual table. |
| 10 | What is the logical order of query execution? | FROM, JOIN, WHERE, GROUP BY, HAVING, SELECT, DISTINCT, ORDER BY, LIMIT. |
| 11 | What are ACID properties? | Atomicity, Consistency, Isolation, Durability. |
| 12 | `WHERE` vs `HAVING`? | WHERE filters rows before grouping; HAVING filters groups after. |